In [1]:
import pandas as pd
import numpy as np
import os
import joblib  # 모델 저장/로드에 사용
from surprise import Reader, Dataset, SVD
from surprise.model_selection import cross_validate, train_test_split
from surprise import accuracy
print("필요한 라이브러리를 모두 임포트했습니다.")

# 1. 파일 경로 설정 및 데이터 로드

# 현재 .ipynb 파일이 있는 'notebooks' 폴더 기준으로 상위 폴더(.)가 기준
BASE_DIR = '../'
OUTPUT_DIR = os.path.join(BASE_DIR, 'output')

# (필수 파일) 전처리된 사용자-아이템-평점 로그 로드
# 경로 설정: 'notebooks'에서 '../output/preprocessed_reviews.csv'
reviews_path = os.path.join(OUTPUT_DIR, 'preprocessed_reviews.csv')

try:
    # preprocessed_reviews.csv (reviewerID, asin, overall) 로드
    reviews_df = pd.read_csv(reviews_path)
    
    # surprise에서 요구하는 컬럼명으로 통일 (user, item, rating)
    # 제안서에 명시된 컬럼명 ('reviewerID', 'asin', 'overall')을 사용합니다.
    df = reviews_df[['reviewerID', 'asin', 'overall']].copy()
    df.columns = ['user', 'item', 'rating']
    
    # 데이터 확인
    print(f"\n평점 데이터 로드 완료. (총 {len(df)}개 로그)")
    print(df.head())

except FileNotFoundError:
    print(f"!!! 에러: 파일 '{reviews_path}'를 찾을 수 없습니다. 경로를 확인해주세요.")
    exit()
except Exception as e:
    print(f"데이터 로드 중 오류 발생: {e}")
    exit()


# 2. surprise 데이터셋 준비

# Reader 객체 정의: 평점 척도는 1.0 ~ 5.0
reader = Reader(rating_scale=(1.0, 5.0))

# DataFrame을 surprise Dataset 형식으로 로드
data = Dataset.load_from_df(df[['user', 'item', 'rating']], reader)
print("\n✅ surprise Dataset 변환 완료.")


# 3. SVD 모델 구축 및 교차 검증

# SVD 모델 정의 (Matrix Factorization 기반)
# n_factors: 잠재 요인 수 (기본 100), random_state로 재현성 확보
model = SVD(random_state=42)

# 5-겹 교차 검증을 통해 모델 성능 평가 (RMSE, MAE)
print("\nSVD 모델 5-Fold 교차 검증 시작 (RMSE / MAE)...")
cv_results = cross_validate(model, data, measures=['RMSE', 'MAE'], cv=5, verbose=False, n_jobs=-1)

# 교차 검증 결과 분석
rmse_mean = np.mean(cv_results['test_rmse'])
mae_mean = np.mean(cv_results['test_mae'])

print("\n--- SVD 모델 교차 검증 결과 ---")
print(f"평균 RMSE (Rating Accuracy): {rmse_mean:.4f}")
print(f"평균 MAE (Rating Accuracy): {mae_mean:.4f}")
print("------------------------------")


# 4. 전체 데이터셋으로 최종 모델 학습 및 예측 함수 정의

# 전체 데이터를 학습용 Trainset으로 생성
trainset = data.build_full_trainset()

# 최종 SVD 모델 학습
final_svd_model = SVD(random_state=42)
final_svd_model.fit(trainset)
print("\n전체 데이터셋으로 최종 SVD 모델 학습 완료.")

# --- 추천/예측 점수 생성 함수 정의 ---
def get_svd_prediction_score(user_id, item_id):
    """
    특정 사용자와 아이템에 대한 SVD 예측 평점(Collaborative Score)을 반환합니다.
    """
    # model.predict(uid, iid, r_ui_known=None, ...).est
    # est가 예측된 평점(Estimated Score)입니다.
    prediction = final_svd_model.predict(user_id, item_id)
    return prediction.est

# --- 함수 사용 예시 ---
# 첫 번째 사용자 ID 가져오기
sample_user_id = trainset.to_raw_uid(0)
# 첫 번째 아이템 ID 가져오기 (콘텐츠 모델 예시와 동일한 'B000000XDJ' 사용)
TEST_ASIN = 'B000000XDJ'

# 예측 점수 계산
svd_score = get_svd_prediction_score(sample_user_id, TEST_ASIN)

print(f"\n[SVD 예측 평점 예시 (Collaborative Score)]")
print(f"사용자 ID: {sample_user_id}")
print(f"아이템 ASIN: {TEST_ASIN}")
print(f"SVD 예측 평점: {svd_score:.4f} / 5.0")


# 5. 모델 저장 (선택)
# 학습된 모델을 저장하여 나중에 하이브리드 모델에서 다시 로드하여 사용
model_save_path = os.path.join(OUTPUT_DIR, 'svd_collaborative_model.joblib')
joblib.dump(final_svd_model, model_save_path)
print(f"\nSVD 모델 저장 완료: {model_save_path}")

필요한 라이브러리를 모두 임포트했습니다.

평점 데이터 로드 완료. (총 1521519개 로그)
                           user        item  rating
0  AFWHJ6O3PV4JC7PVOJH6CPULO2KQ  B07Z76Y18X     5.0
1  AFWHJ6O3PV4JC7PVOJH6CPULO2KQ  B00004NKAK     5.0
2  AFWHJ6O3PV4JC7PVOJH6CPULO2KQ  B0000062P5     5.0
3  AFWHJ6O3PV4JC7PVOJH6CPULO2KQ  B002HMHR7S     4.0
4  AFWHJ6O3PV4JC7PVOJH6CPULO2KQ  B000002AGY     5.0

✅ surprise Dataset 변환 완료.

SVD 모델 5-Fold 교차 검증 시작 (RMSE / MAE)...

--- SVD 모델 교차 검증 결과 ---
평균 RMSE (Rating Accuracy): 0.8412
평균 MAE (Rating Accuracy): 0.5731
------------------------------

전체 데이터셋으로 최종 SVD 모델 학습 완료.

[SVD 예측 평점 예시 (Collaborative Score)]
사용자 ID: AFWHJ6O3PV4JC7PVOJH6CPULO2KQ
아이템 ASIN: B000000XDJ
SVD 예측 평점: 5.0000 / 5.0

SVD 모델 저장 완료: ../output/svd_collaborative_model.joblib
